In [2]:
# ── Install Phase 2 dependencies
# New vs Phase 1: sentencepiece already installed; we add nothing heavy.
# T5-small fits on T4 comfortably alongside the eval LLM.
import subprocess, sys

packages = [
    'rank_bm25',        # BM25 (re-used from Phase 1)
    'spacy',            # Sentence chunker (re-used)
    'datasets',         # HuggingFace datasets
    'bitsandbytes',     # 4-bit quant for Mistral-7B
    'accelerate',       # Device mapping
    'sentencepiece',    # T5 tokenizer
    'matplotlib',
    'seaborn',
    'tqdm',
    'rouge_score',
    'nltk',
    'pyarrow',          # parquet loading
    'fastparquet',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)
print('✅ All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All dependencies installed


In [3]:
import os, re, json, time, warnings, random
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import spacy
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Paths
# Phase 1 outputs live in the input dataset you added on Kaggle
PHASE1_DIR  = Path('/kaggle/input/datasets/zaibunnisaq/phase1')   # ← rename to match your dataset slug
OUTPUT_DIR  = Path('/kaggle/working/apol_phase2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU:    Tesla T4
VRAM:   15.6 GB


In [4]:
# ── Load everything Phase 1 saved

# 1. Config (must match Phase 1 exactly — same LLM, same top-k, same budgets)
with open(PHASE1_DIR / 'phase1_config.json') as f:
    p1_config = json.load(f)

print('Phase 1 config loaded:')
for k, v in p1_config.items():
    print(f'  {k}: {v}')

# 2. Baseline results for comparison
df_baseline = pd.read_csv(PHASE1_DIR / 'phase1_baseline_results.csv')
print(f'\nBaseline results shape: {df_baseline.shape}')
print(df_baseline.to_string(index=False))

# 3. Pre-chunked + retrieved samples (main compressor input)
with open(PHASE1_DIR / 'samples_with_sentences.json') as f:
    samples_data = json.load(f)
print(f'\nSamples loaded: {len(samples_data)}')
print(f'Example keys:   {list(samples_data[0].keys())}')
print(f'Example Q:      {samples_data[0]["question"][:80]}')
print(f'Sentences:      {len(samples_data[0]["sentences"])} retrieved')

# 4. Prompt bank (for LLM eval)
df_prompts = pd.read_parquet(PHASE1_DIR / 'prompt_bank.parquet')
df_prompts['answers'] = df_prompts['answers'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
print(f'\nPrompt bank shape: {df_prompts.shape}')
print(df_prompts[['sample_id','dataset','length_variant','total_tokens']].head(6))

Phase 1 config loaded:
  llm_model_id: mistralai/Mistral-7B-Instruct-v0.2
  llm_load_in_4bit: True
  bm25_top_k: 10
  prompt_lengths: {'short': 512, 'medium': 2048, 'long': 4096}
  hotpotqa_n: 200
  nq_n: 200
  eval_sample_size: 100
  phase1_baseline_em: 0.015
  phase1_baseline_f1: 0.08729869606765808
  phase1_avg_tokens: 254.555

Baseline results shape: (6, 10)
          dataset length_variant  n_samples   em     f1  efficiency_score  avg_total_tokens  avg_context_tok  avg_latency_s  pct_truncated
         hotpotqa          short        100 0.03 0.1146            0.2966            387.10           316.88        15.9022           0.05
         hotpotqa         medium        100 0.03 0.1157            0.2970            390.85           320.58        15.9984           0.00
         hotpotqa           long        100 0.03 0.1157            0.2970            390.85           320.58        15.8351           0.00
natural_questions          short        100 0.00 0.0593            0.4933      

In [5]:
def approx_tokens(text: str) -> int:
    return int(len(text.split()) * 1.3)

STOPWORDS = set(stopwords.words('english'))

class SentenceChunker:
    def __init__(self, model='en_core_web_sm', min_sent_tokens=5, max_sent_tokens=120):
        self.nlp = spacy.load(model, disable=['ner', 'lemmatizer'])
        self.nlp.max_length = 2_000_000
        self.min_tokens = min_sent_tokens
        self.max_tokens = max_sent_tokens

    def chunk_document(self, doc_text: str, doc_idx: int) -> List[Dict]:
        doc_text = doc_text.strip()
        if not doc_text:
            return []
        if len(doc_text) <= 100_000:
            spacy_doc = self.nlp(doc_text)
            raw_sents = [s.text.strip() for s in spacy_doc.sents]
        else:
            raw_sents = re.split(r'(?<=[.!?])\s+', doc_text)
        sentences = []
        for s_idx, sent in enumerate(raw_sents):
            sent = sent.strip()
            n_tokens = approx_tokens(sent)
            if n_tokens < self.min_tokens:
                continue
            if n_tokens > self.max_tokens:
                sub_sents = re.split(r'(?<=[,;:])\s+', sent)
                for sub_idx, sub in enumerate(sub_sents):
                    if approx_tokens(sub) >= self.min_tokens:
                        sentences.append({'text': sub.strip(), 'source_doc': doc_idx,
                                          'sent_idx': f'{s_idx}.{sub_idx}',
                                          'token_count': approx_tokens(sub)})
            else:
                sentences.append({'text': sent, 'source_doc': doc_idx,
                                  'sent_idx': str(s_idx), 'token_count': n_tokens})
        return sentences

    def chunk_sample(self, sample) -> List[Dict]:
        all_sents = []
        for doc_idx, doc in enumerate(sample.context_docs):
            sents = self.chunk_document(doc, doc_idx)
            all_sents.extend(sents)
        return all_sents


class BM25Retriever:
    def __init__(self, top_k=10):
        self.top_k = top_k

    def _tokenize(self, text: str) -> List[str]:
        tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text.lower())
        return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

    def retrieve(self, query: str, sentences: List[Dict]) -> List[Dict]:
        if not sentences:
            return []
        tokenized_corpus = [self._tokenize(s['text']) for s in sentences]
        tokenized_query  = self._tokenize(query)
        bm25   = BM25Okapi(tokenized_corpus)
        scores = bm25.get_scores(tokenized_query)
        ranked = [{**s, 'bm25_score': float(sc), 'original_idx': i}
                  for i, (s, sc) in enumerate(zip(sentences, scores))]
        ranked.sort(key=lambda x: x['bm25_score'], reverse=True)
        top = ranked[:self.top_k]
        top.sort(key=lambda x: (x['source_doc'], x['original_idx']))
        for rank, s in enumerate(top):
            s['rank'] = rank
        return top


chunker   = SentenceChunker()
retriever = BM25Retriever(top_k=p1_config['bm25_top_k'])
print('✅ Chunker and BM25Retriever initialized')

✅ Chunker and BM25Retriever initialized


In [6]:
# ── HardPruner
# Drops sentences whose BM25 relevance score falls below a threshold.
# Three aggressiveness levels correspond to the compression ratio targets.

class HardPruner:
    """
    BM25-score-based sentence pruning.

    Strategy:
      - Sentences are already BM25-ranked (from Phase 1 retrieval).
      - We keep only the top `keep_ratio` fraction by score.
      - A minimum of `min_keep` sentences is always retained (safety floor).
      - Token budget is the final gate: we stop adding sentences once we'd
        exceed `token_budget`.

    This gives us three compression levels:
      conservative (keep_ratio=0.7) → ~30% reduction
      moderate     (keep_ratio=0.5) → ~50% reduction
      aggressive   (keep_ratio=0.3) → ~70% reduction
    """

    LEVELS = {
        'conservative': 0.7,
        'moderate':     0.5,
        'aggressive':   0.3,
    }

    def __init__(self, keep_ratio: float = 0.5, min_keep: int = 2):
        self.keep_ratio = keep_ratio
        self.min_keep   = min_keep

    def prune(self, sentences: List[Dict], token_budget: int = 99999) -> Dict:
        """
        Args:
            sentences:    BM25-ranked sentence dicts (must have 'bm25_score', 'token_count')
            token_budget: max tokens for the context section of the prompt

        Returns dict with:
            kept_sentences:   sentences that survive pruning (doc order restored)
            dropped_sentences: sentences that were cut
            original_tokens:  total tokens before pruning
            kept_tokens:      total tokens after pruning
            compression_ratio: 1 - (kept / original)
            n_kept / n_dropped
        """
        if not sentences:
            return self._empty_result()

        original_tokens = sum(s['token_count'] for s in sentences)
        n_total   = len(sentences)
        n_keep    = max(self.min_keep, int(n_total * self.keep_ratio))

        # Sort by BM25 score descending — pick top n_keep
        by_score = sorted(sentences, key=lambda x: x.get('bm25_score', 0), reverse=True)
        selected = by_score[:n_keep]

        # Apply token budget (greedily add by score until budget exhausted)
        budget_selected, running_tokens = [], 0
        for s in selected:
            if running_tokens + s['token_count'] <= token_budget:
                budget_selected.append(s)
                running_tokens += s['token_count']
            # once over budget, skip — don't break, a shorter later sentence might still fit
            # (conservative: allow small gaps to fill)

        if len(budget_selected) < self.min_keep:
            # Safety: always keep at least min_keep highest-score sentences
            budget_selected = selected[:self.min_keep]
            running_tokens  = sum(s['token_count'] for s in budget_selected)

        # Restore document order (preserve narrative flow)
        kept = sorted(budget_selected, key=lambda x: (x.get('source_doc', 0),
                                                        x.get('original_idx', 0)))

        # Identify dropped sentences
        kept_ids = {id(s) for s in kept}
        dropped  = [s for s in sentences if id(s) not in kept_ids]

        kept_tokens = sum(s['token_count'] for s in kept)
        ratio = 1 - (kept_tokens / original_tokens) if original_tokens > 0 else 0

        return {
            'kept_sentences':    kept,
            'dropped_sentences': dropped,
            'original_tokens':   original_tokens,
            'kept_tokens':       kept_tokens,
            'compression_ratio': ratio,
            'n_kept':            len(kept),
            'n_dropped':         len(dropped),
        }

    def _empty_result(self):
        return {'kept_sentences': [], 'dropped_sentences': [],
                'original_tokens': 0, 'kept_tokens': 0,
                'compression_ratio': 0, 'n_kept': 0, 'n_dropped': 0}


# ── Quick smoke-test
pruner_moderate = HardPruner(keep_ratio=0.5)
test_sents = samples_data[0]['sentences']
result = pruner_moderate.prune(test_sents)

print(f'Sample: {samples_data[0]["question"][:70]}')
print(f'Original sentences: {len(test_sents):>4}  |  tokens: {result["original_tokens"]}')
print(f'Kept:               {result["n_kept"]:>4}  |  tokens: {result["kept_tokens"]}')
print(f'Dropped:            {result["n_dropped"]:>4}')
print(f'Compression ratio:  {result["compression_ratio"]:.2%}')
print('\nKept sentences:')
for s in result['kept_sentences'][:3]:
    print(f'  [score={s.get("bm25_score", 0):.3f}] {s["text"][:90]}')

Sample: Were Scott Derrickson and Ed Wood of the same nationality?
Original sentences:   10  |  tokens: 274
Kept:                  5  |  tokens: 126
Dropped:               5
Compression ratio:  54.01%

Kept sentences:
  [score=4.227] Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed 
  [score=4.677] Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwri
  [score=4.102] Woodson is adjacent to the Wood Plantation, the largest of the plantations own by Ed Wood 


In [7]:
from transformers import T5ForConditionalGeneration, T5TokenizerFast

# ── Load T5-small once — ~240MB, fits easily on T4 alongside Mistral
print('Loading T5-small...')
T5_MODEL_ID = 'google-t5/t5-small'

t5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_ID)
t5_model     = T5ForConditionalGeneration.from_pretrained(
    T5_MODEL_ID, torch_dtype=torch.float16
).to(DEVICE)
t5_model.eval()
print(f'✅ T5-small loaded on {DEVICE}')
print(f'   Params: {sum(p.numel() for p in t5_model.parameters()) / 1e6:.1f}M')


class SoftCompressor:
    """
    Compresses individual sentences using T5-small abstractive summarization.

    Design choices:
      - Input prefix: "summarize: {sentence}" — standard T5 summarization prompt
      - min_length guard: prevents T5 collapsing facts into a single word
      - fallback: if T5 output is LONGER than the input, return original (no bloat)
      - batch processing: handles a list of sentences in one forward pass (GPU efficient)

    This is the EMBED action precursor — in Phase 3, AHCC will learn to call
    this compressor only on sentences it classifies as EMBED.
    """

    def __init__(self,
                 model:     T5ForConditionalGeneration = t5_model,
                 tokenizer: T5TokenizerFast            = t5_tokenizer,
                 max_input_length:  int = 256,
                 max_output_length: int = 64,
                 min_output_length: int = 10,
                 batch_size:        int = 16,
                 device:            str = DEVICE):
        self.model      = model
        self.tokenizer  = tokenizer
        self.max_in     = max_input_length
        self.max_out    = max_output_length
        self.min_out    = min_output_length
        self.batch_size = batch_size
        self.device     = device

    @torch.no_grad()
    def compress_batch(self, texts: List[str]) -> List[str]:
        """Compress a list of text strings. Returns compressed strings (same order)."""
        if not texts:
            return []

        prefixed = [f'summarize: {t}' for t in texts]
        results  = []

        for batch_start in range(0, len(prefixed), self.batch_size):
            batch = prefixed[batch_start: batch_start + self.batch_size]

            inputs = self.tokenizer(
                batch,
                max_length=self.max_in,
                truncation=True,
                padding=True,
                return_tensors='pt'
            ).to(self.device)

            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_out,
                min_new_tokens=self.min_out,
                num_beams=2,           # beam=2: quality/speed balance on T4
                early_stopping=True,
                no_repeat_ngram_size=3,
            )

            decoded = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            results.extend(decoded)

        return results

    def compress_sentences(self, sentences: List[Dict]) -> List[Dict]:
        """
        Takes a list of sentence dicts, compresses each one.
        Returns new list of dicts with 'text' replaced by compressed version
        and 'original_text', 'original_token_count', 'compressed' flag added.
        """
        if not sentences:
            return []

        texts      = [s['text'] for s in sentences]
        compressed = self.compress_batch(texts)

        results = []
        for orig_sent, comp_text in zip(sentences, compressed):
            orig_tokens = orig_sent['token_count']
            comp_tokens = approx_tokens(comp_text)

            # Fallback: if T5 made it longer, keep original
            use_text   = comp_text   if comp_tokens < orig_tokens else orig_sent['text']
            use_tokens = comp_tokens if comp_tokens < orig_tokens else orig_tokens

            results.append({
                **orig_sent,
                'text':                 use_text,
                'token_count':          use_tokens,
                'original_text':        orig_sent['text'],
                'original_token_count': orig_tokens,
                'compressed':           comp_tokens < orig_tokens,
                'compression_ratio':    1 - (use_tokens / orig_tokens) if orig_tokens > 0 else 0,
            })

        return results


# ── Smoke-test
compressor = SoftCompressor()

test_sentences = test_sents[:5]
compressed_sents = compressor.compress_sentences(test_sentences)

print('Soft Compression Smoke Test')
print('='*70)
for orig, comp in zip(test_sentences[:3], compressed_sents[:3]):
    print(f'ORIGINAL  ({orig["token_count"]:>3}t): {orig["text"][:100]}')
    print(f'COMPRESSED({comp["token_count"]:>3}t): {comp["text"][:100]}')
    print(f'Ratio: {comp["compression_ratio"]:.2%}  |  Fallback used: {not comp["compressed"]}')
    print('-'*70)

Loading T5-small...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ T5-small loaded on cuda
   Params: 60.5M
Soft Compression Smoke Test
ORIGINAL  ( 36t): Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produc
COMPRESSED( 19t): a 1994 American biographical period comedy-drama film. starring Johnny Depp as cult filmmaker Ed Woo
Ratio: 47.22%  |  Fallback used: False
----------------------------------------------------------------------
ORIGINAL  ( 19t): Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and pr
COMPRESSED( 11t): Scott Derrickson is an american director, screenwriter and producer.
Ratio: 42.11%  |  Fallback used: False
----------------------------------------------------------------------
ORIGINAL  ( 39t): Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a promin
COMPRESSED( 39t): Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a promin
Ratio: 0.00%  |  Fallback u

In [8]:
class HybridPipeline:
    """
    BM25 hard prune → T5 soft compress → verbatim KEEP

    Decision logic per sentence (mirrors AHCC's KEEP/DROP/EMBED):
      score >= keep_threshold   → KEEP   (high relevance: verbatim)
      score >= embed_threshold  → EMBED  (medium relevance: T5 compress)
      score <  embed_threshold  → DROP   (low relevance: remove)

    Thresholds are on normalised BM25 score (0–1 scale).
    These hand-crafted thresholds become heuristic labels for Phase 3 AHCC training.
    """

    # These are logged to phase2_config.json so Phase 3 can use them as label rules
    DEFAULT_KEEP_THRESHOLD  = 0.6   # top 40% by score → KEEP
    DEFAULT_EMBED_THRESHOLD = 0.25  # mid 35% → EMBED, bottom 25% → DROP

    def __init__(self,
                 compressor:       SoftCompressor,
                 keep_threshold:   float = DEFAULT_KEEP_THRESHOLD,
                 embed_threshold:  float = DEFAULT_EMBED_THRESHOLD,
                 token_budget:     int   = 99999):
        self.compressor      = compressor
        self.keep_threshold  = keep_threshold
        self.embed_threshold = embed_threshold
        self.token_budget    = token_budget

    def _normalise_scores(self, sentences: List[Dict]) -> List[Dict]:
        """Min-max normalise BM25 scores to [0, 1]."""
        scores = [s.get('bm25_score', 0) for s in sentences]
        min_s, max_s = min(scores), max(scores)
        rng = max_s - min_s if max_s > min_s else 1.0
        return [{**s, 'norm_score': (s.get('bm25_score', 0) - min_s) / rng}
                for s in sentences]

    def run(self, sentences: List[Dict]) -> Dict:
        """
        Returns:
            kept_verbatim:   KEEP sentences (verbatim)
            embedded:        EMBED sentences after T5 compression
            dropped:         DROP sentences
            all_kept:        kept_verbatim + embedded (for prompt assembly, doc order)
            original_tokens
            final_tokens
            compression_ratio
            action_counts:   {'KEEP': n, 'EMBED': n, 'DROP': n}
            label_records:   list of (sentence, action) — heuristic labels for Phase 3
        """
        if not sentences:
            return self._empty_result()

        original_tokens = sum(s['token_count'] for s in sentences)
        normed = self._normalise_scores(sentences)

        keep_sents  = [s for s in normed if s['norm_score'] >= self.keep_threshold]
        embed_sents = [s for s in normed if self.embed_threshold <= s['norm_score'] < self.keep_threshold]
        drop_sents  = [s for s in normed if s['norm_score'] < self.embed_threshold]

        # Compress the EMBED bucket
        embedded_compressed = self.compressor.compress_sentences(embed_sents) if embed_sents else []

        # Apply token budget: KEEP first (highest priority), then compressed EMBED
        all_candidates = (
            [(s, 'KEEP')  for s in keep_sents] +
            [(s, 'EMBED') for s in embedded_compressed]
        )
        # Sort by norm_score descending for budget allocation
        all_candidates.sort(key=lambda x: x[0].get('norm_score', 0), reverse=True)

        selected, running_tokens = [], 0
        for s, action in all_candidates:
            t = s['token_count']
            if running_tokens + t <= self.token_budget:
                selected.append((s, action))
                running_tokens += t

        # Restore doc order
        selected.sort(key=lambda x: (x[0].get('source_doc', 0), x[0].get('original_idx', 0)))

        kept_verbatim = [s for s, a in selected if a == 'KEEP']
        embedded_out  = [s for s, a in selected if a == 'EMBED']
        all_kept      = [s for s, _ in selected]

        final_tokens = sum(s['token_count'] for s in all_kept)
        ratio = 1 - (final_tokens / original_tokens) if original_tokens > 0 else 0

        # Build heuristic label records for Phase 3
        label_records = (
            [{'text': s['text'], 'norm_score': s.get('norm_score', 0),
              'bm25_score': s.get('bm25_score', 0), 'action': 'KEEP'} for s in keep_sents] +
            [{'text': s.get('original_text', s['text']), 'norm_score': s.get('norm_score', 0),
              'bm25_score': s.get('bm25_score', 0), 'action': 'EMBED'} for s in embed_sents] +
            [{'text': s['text'], 'norm_score': s.get('norm_score', 0),
              'bm25_score': s.get('bm25_score', 0), 'action': 'DROP'}  for s in drop_sents]
        )

        return {
            'kept_verbatim':    kept_verbatim,
            'embedded':         embedded_out,
            'dropped':          drop_sents,
            'all_kept':         all_kept,
            'original_tokens':  original_tokens,
            'final_tokens':     final_tokens,
            'compression_ratio': ratio,
            'action_counts':    {'KEEP': len(keep_sents), 'EMBED': len(embed_sents),
                                 'DROP': len(drop_sents)},
            'label_records':    label_records,
        }

    def _empty_result(self):
        return {'kept_verbatim': [], 'embedded': [], 'dropped': [], 'all_kept': [],
                'original_tokens': 0, 'final_tokens': 0, 'compression_ratio': 0,
                'action_counts': {'KEEP': 0, 'EMBED': 0, 'DROP': 0}, 'label_records': []}


# ── Smoke test
hybrid = HybridPipeline(compressor=compressor)
h_result = hybrid.run(test_sents)

print(f'Hybrid Pipeline Smoke Test')
print(f'Original tokens:  {h_result["original_tokens"]}')
print(f'Final tokens:     {h_result["final_tokens"]}')
print(f'Compression:      {h_result["compression_ratio"]:.2%}')
print(f'Action counts:    {h_result["action_counts"]}')
print(f'\nLabel records sample (first 4):')
for r in h_result['label_records'][:4]:
    print(f'  [{r["action"]:4}] score={r["norm_score"]:.2f} | {r["text"][:80]}')

Hybrid Pipeline Smoke Test
Original tokens:  274
Final tokens:     106
Compression:      61.31%
Action counts:    {'KEEP': 3, 'EMBED': 2, 'DROP': 5}

Label records sample (first 4):
  [KEEP] score=0.74 | Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film
  [KEEP] score=1.00 | Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director,
  [KEEP] score=0.67 | Woodson is adjacent to the Wood Plantation, the largest of the plantations own b
  [EMBED] score=0.29 | Collis first work was the assistant director for the Scott Derrickson's short "L


In [9]:
# ── PromptConstructor — re-used from Phase 1 with minor extension
# The only change: accepts pre-processed sentence lists instead of raw docs.

PROMPT_TEMPLATE = (
    "[INST] Answer the question using only the provided context. "
    "Be concise — answer in as few words as possible.\n\n"
    "Question: {question}\n\n"
    "Context:\n{context}\n\n"
    "Answer: [/INST]"
)

def sentences_to_context(sentences: List[Dict]) -> str:
    """Joins kept sentences back into a context string (doc order preserved)."""
    return ' '.join(s['text'] for s in sentences)

def build_prompt(question: str, context: str, token_budget: int = 2048) -> Tuple[str, int, bool]:
    """
    Returns (prompt_string, token_count, was_truncated).
    Truncates context word-by-word if over budget.
    """
    template_overhead = approx_tokens(
        PROMPT_TEMPLATE.replace('{question}', question).replace('{context}', '')
    )
    context_budget = token_budget - template_overhead - 20  # 20-token safety margin

    ctx_words   = context.split()
    ctx_tokens  = approx_tokens(context)
    truncated   = False

    if ctx_tokens > context_budget:
        # Truncate context to fit
        target_words = int(context_budget / 1.3)
        ctx_words    = ctx_words[:target_words]
        context      = ' '.join(ctx_words)
        truncated    = True

    prompt      = PROMPT_TEMPLATE.format(question=question, context=context)
    total_tokens = approx_tokens(prompt)
    return prompt, total_tokens, truncated

print('✅ Prompt constructor ready')

✅ Prompt constructor ready


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL_ID = p1_config['llm_model_id']
LOAD_4BIT    = p1_config['llm_load_in_4bit']

print(f'Loading LLM: {LLM_MODEL_ID}')
print(f'4-bit:       {LOAD_4BIT}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if LOAD_4BIT else None

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = 'left'  

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16 if not LOAD_4BIT else None,
    trust_remote_code=True,
)
llm_model.eval()
print(f'✅ LLM loaded')


@torch.no_grad()
def generate_answer(prompts: List[str], max_new_tokens: int = 128, batch_size: int = 4) -> List[str]:
    """Batch inference. Returns decoded answer strings (stripped)."""
    all_outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i: i + batch_size]
        enc   = llm_tokenizer(batch, return_tensors='pt', padding=True,
                               truncation=True, max_length=4096).to(DEVICE)
        out   = llm_model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
        # Decode only the newly generated tokens
        for j, seq in enumerate(out):
            input_len = enc['input_ids'].shape[1]
            new_tokens = seq[input_len:]
            text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            all_outputs.append(text)
    return all_outputs

print('generate_answer() ready')

Loading LLM: mistralai/Mistral-7B-Instruct-v0.2
4-bit:       True


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ LLM loaded
generate_answer() ready


In [11]:
# ── Exact copies from Phase 1 — do NOT modify (results must be comparable)

def normalize_answer(s: str) -> str:
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[^a-z0-9\s]', '', s)
    return ' '.join(s.split())

def compute_em(prediction: str, gold_answers: List[str]) -> float:
    pred_norm = normalize_answer(prediction)
    return float(any(pred_norm == normalize_answer(g) for g in gold_answers))

def compute_f1(prediction: str, gold_answers: List[str]) -> float:
    pred_tokens = normalize_answer(prediction).split()
    best_f1 = 0.0
    for gold in gold_answers:
        gold_tokens = normalize_answer(gold).split()
        common = Counter(pred_tokens) & Counter(gold_tokens)
        n_common = sum(common.values())
        if n_common == 0:
            continue
        precision = n_common / len(pred_tokens)
        recall    = n_common / len(gold_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best_f1 = max(best_f1, f1)
    return best_f1

def compute_efficiency_score(f1: float, total_tokens: int) -> float:
    return f1 / max(total_tokens / 1000, 0.1)

print('✅ Metrics ready (identical to Phase 1)')

✅ Metrics ready (identical to Phase 1)


In [12]:
# ── Re-retrieve with higher top-k for long variant
# samples_with_sentences.json only has top-10; long needs top-20

from datasets import load_dataset
from dataclasses import dataclass, field

@dataclass
class QASample:
    sample_id:    str
    question:     str
    answers:      list
    context_docs: list
    dataset:      str
    difficulty:   str = 'medium'
    metadata:     dict = field(default_factory=dict)

# Load raw datasets (same split/size as Phase 1)
print('Reloading HotpotQA...')
raw_hotpot = load_dataset('hotpot_qa', 'distractor', split='validation', trust_remote_code=True)
print('Reloading NQ...')
raw_nq = load_dataset('nq_open', split='validation', trust_remote_code=True)

retriever_long = BM25Retriever(top_k=20)   # 2× Phase 1 top-k

long_samples = {}   # sample_id → sentences (top-20)

# HotpotQA
for i, row in enumerate(raw_hotpot):
    if i >= p1_config['hotpotqa_n']:
        break
    docs = [f"{t}: {'  '.join(s)}" for t, s in
            zip(row['context']['title'], row['context']['sentences'])]
    sample = QASample(
        sample_id=f'hotpot_{row["id"]}', question=row['question'],
        answers=[row['answer']], context_docs=docs, dataset='hotpotqa'
    )
    sents    = chunker.chunk_sample(sample)
    retrieved = retriever_long.retrieve(sample.question, sents)
    long_samples[sample.sample_id] = {
        'sample_id': sample.sample_id, 'dataset': sample.dataset,
        'question':  sample.question,  'answers': sample.answers,
        'sentences': retrieved
    }

# NQ
nq_idx = 0
for i, row in enumerate(raw_nq):
    if nq_idx >= p1_config['nq_n']:
        break
    answers = [a for a in row['answer'] if isinstance(a, str) and a.strip()]
    if not answers:
        continue
    sample_id = f'nq_{i:05d}'
    q_words   = row['question'].replace('?','').split()
    docs = [
        f"According to Wikipedia: {answers[0]}.",
        f"This article discusses topics related to {' '.join(q_words[:4])} and related subjects.",
        f"There are several aspects of {' '.join(q_words[-3:])} that scholars have debated.",
        f"Historical records show varying accounts regarding {' '.join(q_words[1:5])}.",
        f"The geographical and cultural context of {q_words[0] if q_words else 'this topic'} is complex.",
    ]
    random.shuffle(docs)
    sample = QASample(sample_id=sample_id, question=row['question'],
                      answers=answers, context_docs=docs, dataset='natural_questions')
    sents     = chunker.chunk_sample(sample)
    retrieved = retriever_long.retrieve(sample.question, sents)
    long_samples[sample_id] = {
        'sample_id': sample_id, 'dataset': 'natural_questions',
        'question':  row['question'], 'answers': answers,
        'sentences': retrieved
    }
    nq_idx += 1

print(f'Long-variant samples built: {len(long_samples)}')
print(f'Avg sentences (long): {np.mean([len(v["sentences"]) for v in long_samples.values()]):.1f}')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Reloading HotpotQA...


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

...

nq_open/train-00000-of-00001.parquet:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

nq_open/validation-00000-of-00001.parque(…):   0%|          | 0.00/214k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87925 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3610 [00:00<?, ? examples/s]

Long-variant samples built: 400
Avg sentences (long): 12.5


In [14]:
#cell 11
PROMPT_LENGTHS = p1_config['prompt_lengths']
EVAL_N         = p1_config['eval_sample_size']
VARIANT_TOPK   = {'short': 5, 'medium': 10, 'long': None}
eval_samples   = samples_data[:EVAL_N]
all_label_records = []
all_rows = []   # accumulates across all method cells

def run_method(method_name, compressor_obj):
    """Run one method across all length variants. Returns rows."""
    rows = []
    print(f'\n{"="*60}')
    print(f'METHOD: {method_name}')
    print(f'{"="*60}')

    for length_variant, token_budget in PROMPT_LENGTHS.items():
        prompts, meta = [], []

        for sample in tqdm(eval_samples, desc=f'{method_name}/{length_variant}', leave=False):
            top_k = VARIANT_TOPK[length_variant]
            if length_variant == 'long':
                sid   = sample['sample_id']
                sents = long_samples[sid]['sentences'] if sid in long_samples else sample['sentences']
            else:
                sents = sample['sentences'][:top_k] if top_k else sample['sentences']

            q      = sample['question']
            answer = sample['answers']

            if 'hard_prune' in method_name:
                result   = compressor_obj.prune(sents, token_budget=token_budget)
                kept     = result['kept_sentences']
                orig_tok = result['original_tokens']
                kept_tok = result['kept_tokens']
                comp_r   = result['compression_ratio']
                actions  = {'KEEP': result['n_kept'], 'DROP': result['n_dropped'], 'EMBED': 0}

            elif method_name == 'soft_compress':
                compressed_sents = compressor.compress_sentences(sents)
                kept     = compressed_sents
                orig_tok = sum(s['token_count'] for s in sents)
                kept_tok = sum(s['token_count'] for s in compressed_sents)
                comp_r   = 1 - (kept_tok / orig_tok) if orig_tok > 0 else 0
                actions  = {'KEEP': 0, 'EMBED': len(sents), 'DROP': 0}

            elif method_name == 'hybrid':
                result   = compressor_obj.run(sents)
                kept     = result['all_kept']
                orig_tok = result['original_tokens']
                kept_tok = result['final_tokens']
                comp_r   = result['compression_ratio']
                actions  = result['action_counts']
                for rec in result['label_records']:
                    rec['sample_id'] = sample['sample_id']
                    rec['dataset']   = sample['dataset']
                    rec['question']  = q
                all_label_records.extend(result['label_records'])

            context   = sentences_to_context(kept)
            prompt, total_tok, truncated = build_prompt(q, context, token_budget)
            prompts.append(prompt)
            meta.append({
                'sample_id': sample['sample_id'], 'dataset': sample['dataset'],
                'question': q, 'answers': answer,
                'length_variant': length_variant, 'method': method_name,
                'original_tokens': orig_tok, 'context_tokens': kept_tok,
                'total_tokens': total_tok, 'compression_ratio': comp_r,
                'truncated': truncated,
                'n_keep': actions.get('KEEP', 0),
                'n_embed': actions.get('EMBED', 0),
                'n_drop': actions.get('DROP', 0),
            })

        # Inference in chunks with progress
        print(f'  [{length_variant}] Running inference on {len(prompts)} prompts...')
        predictions = []
        t0 = time.time()
        for i in tqdm(range(0, len(prompts), 4), desc='batches'):
            batch = prompts[i:i+4]
            predictions.extend(generate_answer(batch, max_new_tokens=p1_config.get('llm_max_new_tokens', 128), batch_size=4))
        elapsed = time.time() - t0

        for m, pred in zip(meta, predictions):
            golds = m['answers'] if isinstance(m['answers'], list) else [m['answers']]
            em  = compute_em(pred, golds)
            f1  = compute_f1(pred, golds)
            eff = compute_efficiency_score(f1, m['total_tokens'])
            rows.append({**m, 'prediction': pred, 'em': em, 'f1': f1,
                         'efficiency_score': eff, 'latency_s': elapsed/len(prompts)})

        avg_f1  = np.mean([r['f1']  for r in rows if r['length_variant'] == length_variant])
        avg_comp = np.mean([r['compression_ratio'] for r in rows if r['length_variant'] == length_variant])
        avg_tok = np.mean([r['total_tokens'] for r in rows if r['length_variant'] == length_variant])
        print(f'  [{length_variant:6}] avg F1={avg_f1:.4f}  comp={avg_comp:.2%}  tokens={avg_tok:.0f}  time={elapsed/60:.1f}min')

    return rows


def save_method_results(method_name, rows):
    """Save this method's rows immediately."""
    df = pd.DataFrame(rows)
    path = OUTPUT_DIR / f'results_{method_name}.parquet'
    df.to_parquet(path, index=False)
    print(f'✅ Saved {len(df)} rows → {path.name}')
    return df


print('✅ Setup ready — run method cells below one at a time')

✅ Setup ready — run method cells below one at a time


In [15]:
rows_hpc = run_method('hard_prune_conservative',
                       HardPruner(keep_ratio=HardPruner.LEVELS['conservative']))
save_method_results('hard_prune_conservative', rows_hpc)
all_rows.extend(rows_hpc)


METHOD: hard_prune_conservative


hard_prune_conservative/short:   0%|          | 0/100 [00:00<?, ?it/s]

  [short] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [short ] avg F1=0.1327  comp=40.60%  tokens=145  time=9.4min


...

  [medium] avg F1=0.1970  comp=31.28%  tokens=270  time=8.2min


hard_prune_conservative/long:   0%|          | 0/100 [00:00<?, ?it/s]

  [long] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [long  ] avg F1=0.2029  comp=29.22%  tokens=502  time=9.5min
✅ Saved 300 rows → results_hard_prune_conservative.parquet


In [16]:
rows_hpa = run_method('hard_prune_aggressive',
                       HardPruner(keep_ratio=HardPruner.LEVELS['aggressive']))
save_method_results('hard_prune_aggressive', rows_hpa)
all_rows.extend(rows_hpa)
# Auto download link
display(FileLink(str(OUTPUT_DIR / 'results_hard_prune_aggressive.parquet'),
                 result_html_prefix='⬇️ Download: '))


METHOD: hard_prune_aggressive


hard_prune_aggressive/short:   0%|          | 0/100 [00:00<?, ?it/s]

  [short] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [short ] avg F1=0.1553  comp=59.47%  tokens=115  time=8.5min


...

hard_prune_aggressive/long:   0%|          | 0/100 [00:00<?, ?it/s]

  [long] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [long  ] avg F1=0.2148  comp=70.55%  tokens=238  time=8.0min
✅ Saved 300 rows → results_hard_prune_aggressive.parquet


NameError: name 'FileLink' is not defined

In [ ]:
# ── Build master comparison table: Phase 1 baseline + all Phase 2 methods

# Phase 1 baseline rows — tag them so they appear in the same table
df_b = df_baseline.copy()
df_b['method'] = 'no_compression'
df_b.rename(columns={'avg_total_tokens': 'total_tokens',
                      'avg_context_tok':  'context_tokens'}, inplace=True)
df_b['compression_ratio'] = 0.0

# Summarise Phase 2 results
groupby_cols = ['method', 'dataset', 'length_variant']
summary_p2 = df_results.groupby(groupby_cols).agg(
    n_samples        = ('em', 'count'),
    em               = ('em', 'mean'),
    f1               = ('f1', 'mean'),
    efficiency_score = ('efficiency_score', 'mean'),
    avg_total_tokens = ('total_tokens', 'mean'),
    avg_comp_ratio   = ('compression_ratio', 'mean'),
    avg_latency_s    = ('latency_s', 'mean'),
    pct_truncated    = ('truncated', 'mean'),
).round(4).reset_index()

# Add baseline F1 delta column
baseline_f1_lookup = df_baseline.set_index(['dataset', 'length_variant'])['f1'].to_dict()
summary_p2['baseline_f1'] = summary_p2.apply(
    lambda r: baseline_f1_lookup.get((r['dataset'], r['length_variant']), np.nan), axis=1
)
summary_p2['f1_delta'] = (summary_p2['f1'] - summary_p2['baseline_f1']).round(4)

# Order for display
length_order = ['short', 'medium', 'long']
method_order = ['no_compression', 'hard_prune_conservative', 'hard_prune_moderate',
                'hard_prune_aggressive', 'soft_compress', 'hybrid']

summary_p2['length_variant'] = pd.Categorical(summary_p2['length_variant'],
                                               categories=length_order, ordered=True)
summary_p2 = summary_p2.sort_values(['dataset', 'method', 'length_variant'])

print('='*100)
print('APOL-BENCH PHASE 2 — COMPRESSION RESULTS vs. BASELINE')
print('='*100)

for dataset in ['hotpotqa', 'natural_questions']:
    print(f'\nDataset: {dataset.upper()}')
    print(f'{"Method":<28} {"Variant":<10} {"EM":>6} {"F1":>6} {"F1Δ":>6} '
          f'{"Eff":>6} {"Tokens":>8} {"CompRatio":>10}')
    print('-'*90)
    subset = summary_p2[summary_p2['dataset'] == dataset]
    for _, row in subset.iterrows():
        delta_str = f'{row["f1_delta"]:+.4f}' if not np.isnan(row['f1_delta']) else '  N/A '
        print(f'{row["method"]:<28} {row["length_variant"]:<10} '
              f'{row["em"]:>6.4f} {row["f1"]:>6.4f} {delta_str:>6} '
              f'{row["efficiency_score"]:>6.4f} {int(row["avg_total_tokens"]):>8} '
              f'{row["avg_comp_ratio"]:>9.2%}')

print('='*100)

# Save summary
summary_p2.to_csv(OUTPUT_DIR / 'phase2_summary.csv', index=False)
df_results.to_parquet(OUTPUT_DIR / 'phase2_results_full.parquet', index=False)
print(f'\nSaved phase2_summary.csv and phase2_results_full.parquet')

In [17]:
rows_hpm = run_method('hard_prune_moderate',
                       HardPruner(keep_ratio=HardPruner.LEVELS['moderate']))
save_method_results('hard_prune_moderate', rows_hpm)
all_rows.extend(rows_hpm)


METHOD: hard_prune_moderate


hard_prune_moderate/short:   0%|          | 0/100 [00:00<?, ?it/s]

  [short] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [short ] avg F1=0.1553  comp=59.47%  tokens=115  time=8.5min


...

  [medium] avg F1=0.2149  comp=50.75%  tokens=208  time=7.3min


hard_prune_moderate/long:   0%|          | 0/100 [00:00<?, ?it/s]

  [long] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [long  ] avg F1=0.1965  comp=50.31%  tokens=368  time=8.9min
✅ Saved 300 rows → results_hard_prune_moderate.parquet


In [18]:
rows_sc = run_method('soft_compress', None)
save_method_results('soft_compress', rows_sc)
all_rows.extend(rows_sc)


METHOD: soft_compress


soft_compress/short:   0%|          | 0/100 [00:00<?, ?it/s]

  [short] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [short ] avg F1=0.1408  comp=30.69%  tokens=160  time=8.4min


...

  [medium] avg F1=0.1647  comp=30.58%  tokens=272  time=9.0min


soft_compress/long:   0%|          | 0/100 [00:00<?, ?it/s]

  [long] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [long  ] avg F1=0.1630  comp=30.06%  tokens=497  time=10.4min
✅ Saved 300 rows → results_soft_compress.parquet


In [19]:
rows_hyb = run_method('hybrid',
                       HybridPipeline(compressor=compressor))
save_method_results('hybrid', rows_hyb)
all_rows.extend(rows_hyb)


METHOD: hybrid


hybrid/short:   0%|          | 0/100 [00:00<?, ?it/s]

  [short] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [short ] avg F1=0.1503  comp=58.17%  tokens=117  time=8.2min


...

  [medium] avg F1=0.1953  comp=67.85%  tokens=153  time=7.1min


hybrid/long:   0%|          | 0/100 [00:00<?, ?it/s]

  [long] Running inference on 100 prompts...


batches:   0%|          | 0/25 [00:00<?, ?it/s]

  [long  ] avg F1=0.1989  comp=68.76%  tokens=246  time=8.6min
✅ Saved 300 rows → results_hybrid.parquet


In [20]:
# If session died mid-way, reload from saved files instead of all_rows
saved_files = list(OUTPUT_DIR.glob('results_*.parquet'))
if len(saved_files) > 0:
    df_results = pd.concat([pd.read_parquet(f) for f in sorted(saved_files)], ignore_index=True)
    print(f'Loaded from {len(saved_files)} saved files: {len(df_results)} total rows')
else:
    df_results = pd.DataFrame(all_rows)
    print(f'Built from memory: {len(df_results)} total rows')

df_results.to_parquet(OUTPUT_DIR / 'phase2_results_full.parquet', index=False)
print('✅ phase2_results_full.parquet saved')
print(df_results.groupby(['method','length_variant'])[['f1','compression_ratio','total_tokens']].mean().round(4))

Loaded from 5 saved files: 1500 total rows
✅ phase2_results_full.parquet saved
                                            f1  compression_ratio  \
method                  length_variant                              
hard_prune_aggressive   long            0.2148             0.7055   
                        medium          0.1959             0.7034   
                        short           0.1553             0.5947   
hard_prune_conservative long            0.2029             0.2922   
                        medium          0.1970             0.3128   
                        short           0.1327             0.4060   
hard_prune_moderate     long            0.1965             0.5031   
                        medium          0.2149             0.5075   
                        short           0.1553             0.5947   
hybrid                  long            0.1989             0.6876   
                        medium          0.1953             0.6785   
                        

In [22]:
# ── Cell 12: Build summary_p2 (handles fresh run + session restart)

# Load df_baseline if not in memory
try:
    df_baseline
except NameError:
    df_baseline = pd.read_csv(PHASE1_DIR / 'phase1_baseline_results.csv')
    print('df_baseline reloaded from disk')

# Load df_results — from memory if available, else from saved parquet files
try:
    df_results
    print(f'df_results already in memory: {len(df_results)} rows')
except NameError:
    saved_files = list(OUTPUT_DIR.glob('results_*.parquet'))
    if not saved_files:
        raise FileNotFoundError('No saved result files found in OUTPUT_DIR')
    df_results = pd.concat([pd.read_parquet(f) for f in sorted(saved_files)], ignore_index=True)
    print(f'df_results reloaded from {len(saved_files)} files: {len(df_results)} rows')

# ── Build summary
groupby_cols = ['method', 'dataset', 'length_variant']
summary_p2 = df_results.groupby(groupby_cols).agg(
    n_samples        = ('em', 'count'),
    em               = ('em', 'mean'),
    f1               = ('f1', 'mean'),
    efficiency_score = ('efficiency_score', 'mean'),
    avg_total_tokens = ('total_tokens', 'mean'),
    avg_comp_ratio   = ('compression_ratio', 'mean'),
    avg_latency_s    = ('latency_s', 'mean'),
    pct_truncated    = ('truncated', 'mean'),
).round(4).reset_index()

baseline_f1_lookup = df_baseline.set_index(['dataset', 'length_variant'])['f1'].to_dict()
summary_p2['baseline_f1'] = summary_p2.apply(
    lambda r: baseline_f1_lookup.get((r['dataset'], r['length_variant']), np.nan), axis=1
)
summary_p2['f1_delta'] = (summary_p2['f1'] - summary_p2['baseline_f1']).round(4)

method_order = ['no_compression', 'hard_prune_conservative', 'hard_prune_moderate',
                'hard_prune_aggressive', 'soft_compress', 'hybrid']
L_ORDER      = ['short', 'medium', 'long']

summary_p2['length_variant'] = pd.Categorical(summary_p2['length_variant'],
                                               categories=L_ORDER, ordered=True)
summary_p2 = summary_p2.sort_values(['dataset', 'method', 'length_variant'])

# ── Print comparison table
print('\n' + '='*100)
print('APOL-BENCH PHASE 2 — COMPRESSION RESULTS vs. BASELINE')
print('='*100)

for dataset in ['hotpotqa', 'natural_questions']:
    print(f'\nDataset: {dataset.upper()}')
    print(f'{"Method":<28} {"Variant":<10} {"EM":>6} {"F1":>6} {"F1Δ":>7} '
          f'{"Eff":>6} {"Tokens":>8} {"CompRatio":>10}')
    print('-'*90)
    for _, row in summary_p2[summary_p2['dataset'] == dataset].iterrows():
        delta_str = f'{row["f1_delta"]:+.4f}' if not np.isnan(row['f1_delta']) else '   N/A'
        print(f'{row["method"]:<28} {row["length_variant"]:<10} '
              f'{row["em"]:>6.4f} {row["f1"]:>6.4f} {delta_str:>7} '
              f'{row["efficiency_score"]:>6.4f} {int(row["avg_total_tokens"]):>8} '
              f'{row["avg_comp_ratio"]:>9.2%}')

print('='*100)

# Save
summary_p2.to_csv(OUTPUT_DIR / 'phase2_summary.csv', index=False)
df_results.to_parquet(OUTPUT_DIR / 'phase2_results_full.parquet', index=False)
print('\n✅ Saved phase2_summary.csv and phase2_results_full.parquet')

df_results already in memory: 1500 rows

APOL-BENCH PHASE 2 — COMPRESSION RESULTS vs. BASELINE

Dataset: HOTPOTQA
Method                       Variant        EM     F1     F1Δ    Eff   Tokens  CompRatio
------------------------------------------------------------------------------------------
hard_prune_aggressive        short      0.0300 0.1553 +0.0407 1.3437      114    59.47%
hard_prune_aggressive        medium     0.0200 0.1959 +0.0802 1.3451      145    70.34%
hard_prune_aggressive        long       0.0300 0.2148 +0.0991 0.9086      238    70.55%
hard_prune_conservative      short      0.0200 0.1327 +0.0181 0.9218      144    40.60%
hard_prune_conservative      medium     0.0300 0.1970 +0.0813 0.7324      269    31.28%
hard_prune_conservative      long       0.0300 0.2029 +0.0872 0.4051      502    29.22%
hard_prune_moderate          short      0.0300 0.1553 +0.0407 1.3437      114    59.47%
hard_prune_moderate          medium     0.0400 0.2149 +0.0992 1.0342      208    50.75%
ha

In [24]:
# ── Cell 14: Save heuristic labels & Phase 2 config

# Fix scope — redefine METHODS keys if not in memory
try:
    METHODS
except NameError:
    METHODS = {
        'hard_prune_conservative': None,
        'hard_prune_moderate':     None,
        'hard_prune_aggressive':   None,
        'soft_compress':           None,
        'hybrid':                  None,
    }

# ── Save heuristic labels
if all_label_records:
    df_labels = pd.DataFrame(all_label_records)
    print('Heuristic label distribution (Phase 3 training data):')
    print(df_labels['action'].value_counts())
    print(f'\nTotal label records: {len(df_labels)}')
    print(f'Unique samples:      {df_labels["sample_id"].nunique()}')
    print(f'Score range:         {df_labels["norm_score"].min():.3f} – {df_labels["norm_score"].max():.3f}')
    df_labels.to_parquet(OUTPUT_DIR / 'heuristic_labels.parquet', index=False)
    print(f'\nSaved heuristic_labels.parquet → {len(df_labels)} training pairs for Phase 3')
    n_labels = len(all_label_records)
else:
    print('No label records in memory — checking disk...')
    label_path = OUTPUT_DIR / 'heuristic_labels.parquet'
    if label_path.exists():
        df_labels = pd.read_parquet(label_path)
        n_labels  = len(df_labels)
        print(f'Loaded {n_labels} labels from disk')
    else:
        n_labels = 0
        print('No label records found.')

# ── Phase 2 config snapshot
p2_config = {
    **p1_config,
    'phase2_methods':         list(METHODS.keys()),
    'hybrid_keep_threshold':  HybridPipeline.DEFAULT_KEEP_THRESHOLD,
    'hybrid_embed_threshold': HybridPipeline.DEFAULT_EMBED_THRESHOLD,
    'hard_prune_levels':      HardPruner.LEVELS,
    't5_model_id':            T5_MODEL_ID,
    't5_max_output_length':   64,
    't5_min_output_length':   10,
    'phase2_best_method':     summary_p2.loc[summary_p2['f1'].idxmax(), 'method'],
    'phase2_best_f1':         float(summary_p2['f1'].max()),
    'phase2_avg_compression': float(summary_p2['avg_comp_ratio'].mean()),
    'n_heuristic_labels':     n_labels,
}
with open(OUTPUT_DIR / 'phase2_config.json', 'w') as f:
    json.dump(p2_config, f, indent=2)
print('\nphase2_config.json saved')

# ── List all artifacts
print('\nAll Phase 2 artifacts:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name:<45} {p.stat().st_size/1024:>8.1f} KB')

Heuristic label distribution (Phase 3 training data):
action
DROP     2081
EMBED     795
KEEP      612
Name: count, dtype: int64

Total label records: 3488
Unique samples:      100
Score range:         0.000 – 1.000

Saved heuristic_labels.parquet → 3488 training pairs for Phase 3

phase2_config.json saved

All Phase 2 artifacts:
  heuristic_labels.parquet                         250.0 KB
  phase2_config.json                                 0.9 KB
  phase2_results_full.parquet                      109.6 KB
  phase2_summary.csv                                 1.5 KB
  results_hard_prune_aggressive.parquet             48.4 KB
  results_hard_prune_conservative.parquet           49.8 KB
  results_hard_prune_moderate.parquet               48.1 KB
  results_hybrid.parquet                            48.2 KB
  results_soft_compress.parquet                     49.9 KB


In [25]:
import shutil

# Create a zip file of the entire working/output directory
shutil.make_archive('/kaggle/working/output', 'zip', '/kaggle/working')

print("Zip file created at /kaggle/working/output.zip")

Zip file created at /kaggle/working/output.zip


In [1]:
print(df_results['dataset'].value_counts())
print(df_results['sample_id'].str[:10].value_counts().head())

NameError: name 'df_results' is not defined